# Práctica 04 — Seguimiento de experimentos

**Curso:** MLOps y Monitorización con Python · **Clase 04** · **Tiempo estimado:** 40-55 minutos
**Notebook de clase:** `notebooks_clase/clase04_seguimiento_experimentos.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio parte de una pequeña función incompleta, inspirada directamente en la lógica de comparación y selección que usaste con MLflow en el notebook de clase. Aquí no se ejecuta MLflow de verdad: son funciones de Python pequeñas, para practicar el razonamiento (filtros, umbrales, estabilidad) sin depender de un almacén de experimentos real.
- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. No hace falta ninguna librería externa.

---
## Celda 0 — Setup

In [ ]:
print("Librerías cargadas correctamente (esta práctica no necesita instalar nada nuevo).")

## Ejercicio 1 — ¿Depende la métrica del umbral?

`exactitud` y `f1` se calculan a partir de predicciones ya convertidas en 0/1 (dependen del umbral, normalmente 0,5); `auc` se calcula a partir de probabilidades, sin fijar ningún umbral. Completa `depende_del_umbral(nombre_metrica)` para que devuelva `True` para `"exactitud"` y `"f1"`, y `False` para `"auc"`.

In [ ]:
def depende_del_umbral(nombre_metrica):
    return False      # TODO: devuelve True si nombre_metrica es "exactitud" o "f1", False si es "auc"

# Comprobación automática
assert depende_del_umbral("exactitud") is True, "Completa las líneas marcadas con TODO"
assert depende_del_umbral("f1") is True, "Completa las líneas marcadas con TODO"
assert depende_del_umbral("auc") is False, "Completa las líneas marcadas con TODO"
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — F1 a partir de precisión y exhaustividad

`f1_score` de scikit-learn calcula el F1 internamente, pero la fórmula es simple: la media armónica de precisión y exhaustividad (*recall*), `F1 = 2·P·R / (P+R)`. Completa `f1_desde_precision_recall(precision, exhaustividad)`. Si `precision + exhaustividad == 0`, devuelve `0.0` (para no dividir entre cero, el mismo caso que motiva `zero_division=0` en el notebook de clase).

In [ ]:
def f1_desde_precision_recall(precision, exhaustividad):
    return 0.0      # TODO: devuelve 2 * precision * exhaustividad / (precision + exhaustividad), o 0.0 si el denominador es 0

# Comprobación automática
assert round(f1_desde_precision_recall(0.4, 0.25), 4) == 0.3077, "Completa las líneas marcadas con TODO"
assert f1_desde_precision_recall(1.0, 1.0) == 1.0, "Completa las líneas marcadas con TODO"
assert f1_desde_precision_recall(0.0, 0.0) == 0.0, "sin dividir entre cero"
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — Filtrar configuraciones por un umbral de AUC

Como en la sección 7 del notebook de clase (primer filtro del criterio explícito). Completa `filtrar_por_auc(configuraciones, auc_minimo)`, que recibe una lista de diccionarios con al menos la clave `"auc"`, y devuelve solo los que tienen `auc >= auc_minimo`.

In [ ]:
def filtrar_por_auc(configuraciones, auc_minimo):
    return configuraciones      # TODO: devuelve [c for c in configuraciones if c["auc"] >= auc_minimo]

# Comprobación automática
ejemplo = [{"id": 1, "auc": 0.72}, {"id": 2, "auc": 0.65}, {"id": 3, "auc": 0.70}]
resultado = filtrar_por_auc(ejemplo, 0.70)
assert [c["id"] for c in resultado] == [1, 3], "Completa las líneas marcadas con TODO"
assert filtrar_por_auc(ejemplo, 0.9) == [], "si nadie llega al mínimo, la lista debe quedar vacía"
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Medir la estabilidad entre semillas

Como en la sección 8 del notebook de clase. Completa `desviacion(valores)`, que calcula la desviación estándar poblacional de una lista de números: la raíz cuadrada del promedio de las diferencias al cuadrado respecto a la media (sin usar NumPy ni `statistics`, con un bucle o una comprensión de listas).

In [ ]:
def desviacion(valores):
    return 0.0      # TODO: calcula media = sum(valores)/len(valores) y devuelve (sum((v - media) ** 2 for v in valores) / len(valores)) ** 0.5

# Comprobación automática
assert desviacion([5, 5, 5]) == 0.0, "Completa las líneas marcadas con TODO"
assert round(desviacion([0.30, 0.32, 0.29]), 4) == 0.0125, "Completa las líneas marcadas con TODO"
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — El criterio explícito completo

Integra los ejercicios 3 y 4 (puedes usar `filtrar_por_auc`, ya definida arriba). Completa `elegir_mejor(configuraciones, auc_minimo, f1_minimo)`, que debe:

- quedarse solo con las configuraciones con `auc >= auc_minimo`,
- de esas, quedarse solo con las que tienen `f1 >= f1_minimo`,
- devolver la que tenga menor `"coste"` entre las que queden (usa `min(..., key=...)`), o `None` si no queda ninguna.

In [ ]:
def elegir_mejor(configuraciones, auc_minimo, f1_minimo):
    return None      # TODO: filtra por auc_minimo, luego por f1_minimo, y devuelve min(..., key=lambda c: c["coste"]) o None si la lista queda vacía

# Comprobación automática
configs_prueba = [
    {"id": 1, "auc": 0.72, "f1": 0.0, "coste": 20},
    {"id": 3, "auc": 0.72, "f1": 0.32, "coste": 15},
    {"id": 4, "auc": 0.72, "f1": 0.31, "coste": 22},
    {"id": 6, "auc": 0.65, "f1": 0.30, "coste": 10},
]
elegido = elegir_mejor(configs_prueba, 0.70, 0.30)
assert elegido["id"] == 3, "Completa las líneas marcadas con TODO"
assert elegir_mejor(configs_prueba, 0.90, 0.30) is None, "si nadie supera el AUC mínimo, debe devolver None"
print("Ejercicio 5 correcto ✔")

---
## Reto final — combinar criterio y estabilidad

Integra todo lo anterior. Completa `configuracion_mas_estable(resultados_por_semilla, costes, umbral_estabilidad)`, que recibe:

- `resultados_por_semilla`: un diccionario `{config_id: [f1_semilla_1, f1_semilla_2, ...]}`,
- `costes`: un diccionario `{config_id: coste}`,
- `umbral_estabilidad`: la desviación máxima aceptable.

Debe devolver el `config_id` con **mayor F1 promedio** entre los que tengan `desviacion(f1s) <= umbral_estabilidad`, o `None` si ninguno es lo bastante estable. Usa `desviacion` (ejercicio 4).

In [ ]:
def configuracion_mas_estable(resultados_por_semilla, costes, umbral_estabilidad):
    return None      # TODO: calcula, para cada config_id, la desviacion() de sus f1; descarta los que superen umbral_estabilidad; entre los que queden, devuelve el config_id con mayor f1 promedio (o None si no queda ninguno)

# Comprobación automática
resultados = {3: [0.3166, 0.3202, 0.2940], 4: [0.3138, 0.3258, 0.2902], 5: [0.3085, 0.3249, 0.2902]}
costes = {3: 15, 4: 22, 5: 16}
assert configuracion_mas_estable(resultados, costes, 0.02) == 3, "Completa las líneas marcadas con TODO"
assert configuracion_mas_estable(resultados, costes, 0.001) is None, "con un umbral tan estricto, ninguna configuración debería ser lo bastante estable"
print("Reto final correcto ✔")

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
def depende_del_umbral(nombre_metrica):
    return nombre_metrica in ("exactitud", "f1")

# Comprobación automática
assert depende_del_umbral("exactitud") is True, "Completa las líneas marcadas con TODO"
assert depende_del_umbral("f1") is True, "Completa las líneas marcadas con TODO"
assert depende_del_umbral("auc") is False, "Completa las líneas marcadas con TODO"
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
def f1_desde_precision_recall(precision, exhaustividad):
    if precision + exhaustividad == 0:
        return 0.0
    return 2 * precision * exhaustividad / (precision + exhaustividad)

# Comprobación automática
assert round(f1_desde_precision_recall(0.4, 0.25), 4) == 0.3077, "Completa las líneas marcadas con TODO"
assert f1_desde_precision_recall(1.0, 1.0) == 1.0, "Completa las líneas marcadas con TODO"
assert f1_desde_precision_recall(0.0, 0.0) == 0.0, "sin dividir entre cero"
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
def filtrar_por_auc(configuraciones, auc_minimo):
    return [c for c in configuraciones if c["auc"] >= auc_minimo]

# Comprobación automática
ejemplo = [{"id": 1, "auc": 0.72}, {"id": 2, "auc": 0.65}, {"id": 3, "auc": 0.70}]
resultado = filtrar_por_auc(ejemplo, 0.70)
assert [c["id"] for c in resultado] == [1, 3], "Completa las líneas marcadas con TODO"
assert filtrar_por_auc(ejemplo, 0.9) == [], "si nadie llega al mínimo, la lista debe quedar vacía"
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
def desviacion(valores):
    media = sum(valores) / len(valores)
    return (sum((v - media) ** 2 for v in valores) / len(valores)) ** 0.5

# Comprobación automática
assert desviacion([5, 5, 5]) == 0.0, "Completa las líneas marcadas con TODO"
assert round(desviacion([0.30, 0.32, 0.29]), 4) == 0.0125, "Completa las líneas marcadas con TODO"
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
def elegir_mejor(configuraciones, auc_minimo, f1_minimo):
    supera_auc = filtrar_por_auc(configuraciones, auc_minimo)
    supera_f1 = [c for c in supera_auc if c["f1"] >= f1_minimo]
    if not supera_f1:
        return None
    return min(supera_f1, key=lambda c: c["coste"])

# Comprobación automática
configs_prueba = [
    {"id": 1, "auc": 0.72, "f1": 0.0, "coste": 20},
    {"id": 3, "auc": 0.72, "f1": 0.32, "coste": 15},
    {"id": 4, "auc": 0.72, "f1": 0.31, "coste": 22},
    {"id": 6, "auc": 0.65, "f1": 0.30, "coste": 10},
]
elegido = elegir_mejor(configs_prueba, 0.70, 0.30)
assert elegido["id"] == 3, "Completa las líneas marcadas con TODO"
assert elegir_mejor(configs_prueba, 0.90, 0.30) is None, "si nadie supera el AUC mínimo, debe devolver None"
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
def configuracion_mas_estable(resultados_por_semilla, costes, umbral_estabilidad):
    estables = {
        config_id: f1s for config_id, f1s in resultados_por_semilla.items()
        if desviacion(f1s) <= umbral_estabilidad
    }
    if not estables:
        return None
    return max(estables, key=lambda config_id: sum(estables[config_id]) / len(estables[config_id]))

# Comprobación automática
resultados = {3: [0.3166, 0.3202, 0.2940], 4: [0.3138, 0.3258, 0.2902], 5: [0.3085, 0.3249, 0.2902]}
costes = {3: 15, 4: 22, 5: 16}
assert configuracion_mas_estable(resultados, costes, 0.02) == 3, "Completa las líneas marcadas con TODO"
assert configuracion_mas_estable(resultados, costes, 0.001) is None, "con un umbral tan estricto, ninguna configuración debería ser lo bastante estable"
print("Reto final correcto ✔")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto de la clase no te ha quedado claro (almacén local de MLflow, `log_artifact`, criterio explícito, estabilidad entre semillas)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*